<a href="https://colab.research.google.com/github/MudadlaPranay12/Pranay_GenAI_MAY_2026/blob/main/resumeparser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.2 MB/s eta 0:00:00


In [4]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


In [10]:
# ================= IMPORTS =================

import json
import re
import google.generativeai as genai
import typing_extensions as typing
from google.colab import userdata
from PyPDF2 import PdfReader

# ================= API =================

genai.configure(api_key=userdata.get("GEMINI_API_KEYS"))

model = genai.GenerativeModel("gemini-2.5-flash-lite")

# ================= PROMPT =================

resume_text = """
ANJALI SHARMA
Email: anjali.sharma@gmail.com
Phone: 9876543210
LinkedIn: linkedin.com/in/anjali-sharma

Skills: Python, Django, React, MongoDB

Education:
B.Tech CSE IIT Delhi 2024 CGPA 8.9

Experience:
Software Engineering Intern at Flipkart

Projects:
AgriBot using Gemini API
"""

prompt = f"""
Extract resume details.

Return ONLY JSON.

Required keys:
name
email
phone
linkedin_url
education
skills
experience
projects
total_experience_years
summary

Rules:
1. Use null if missing
2. Do not invent values
3. Phone should be +91XXXXXXXXXX
4. Output only JSON

Resume:
{resume_text}
"""

# =========================================================
# WAY 1
# =========================================================

response_1 = model.generate_content(prompt)

print("=========== WAY 1 ===========")
print(response_1.text)

# =========================================================
# WAY 2
# =========================================================

response_2 = model.generate_content(
    prompt,
    generation_config={
        "response_mime_type": "application/json",
        "temperature": 0.2,
        "max_output_tokens": 2048
    }
)

data_2 = json.loads(response_2.text)

EXPECTED_KEYS = [
    "name",
    "email",
    "phone",
    "linkedin_url",
    "education",
    "skills",
    "experience",
    "projects",
    "total_experience_years",
    "summary"
]

for key in EXPECTED_KEYS:
    data_2.setdefault(key, None)

# validation

if data_2.get("email"):
    if "@" not in data_2.get("email"):
        data_2["email"] = None

phone = data_2.get("phone")

if phone:
    digits = re.sub(r"\D", "", phone)

    if len(digits) == 10:
        data_2["phone"] = "+91" + digits

if data_2.get("total_experience_years"):
    if float(data_2.get("total_experience_years")) < 0:
        data_2["total_experience_years"] = 0.0

print("=========== WAY 2 ===========")
print(json.dumps(data_2, indent=2))

# =========================================================
# WAY 3
# =========================================================

class Resume(typing.TypedDict):
    name: str
    email: str
    phone: str
    linkedin_url: str
    education: list
    skills: list
    experience: list
    projects: list
    total_experience_years: float
    summary: str

response_3 = model.generate_content(
    prompt,
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": Resume,
        "temperature": 0.2,
        "max_output_tokens": 2048
    }
)

data_3 = json.loads(response_3.text)

for key in EXPECTED_KEYS:
    data_3.setdefault(key, None)

print("=========== WAY 3 ===========")
print(json.dumps(data_3, indent=2))

# =========================================================
# SUMMARY
# =========================================================

print("\nSUMMARY")
print("Name:", data_3.get("name"))
print("Email:", data_3.get("email"))
print("Phone:", data_3.get("phone"))
print("Skills:", data_3.get("skills"))
print("Experience:", data_3.get("total_experience_years"))

# =========================================================
# PDF RESUME
# =========================================================

pdf_path = "/content/sample_resume.pdf"
reader = PdfReader(pdf_path)

pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text()

print("\n=========== PDF TEXT ===========")
print(pdf_text[:1000])

=========== WAY 1 ===========
```json
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech CSE",
      "institution": "IIT Delhi",
      "year": 2024,
      "cgpa": "8.9"
    }
  ],
  "skills": [
    "Python",
    "Django",
    "React",
    "MongoDB"
  ],
  "experience": [
    {
      "title": "Software Engineering Intern",
      "company": "Flipkart",
      "dates": null,
      "description": null
    }
  ],
  "projects": [
    {
      "name": "AgriBot",
      "description": "using Gemini API",
      "technologies": null
    }
  ],
  "total_experience_years": null,
  "summary": null
}
```
=========== WAY 2 ===========
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech CSE",
      "institution